In [ ]:
# EuroSAT Land Use Classification with ResNet50
# Simple, focused implementation with Gradio UI

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import BytesIO
import gradio as gr

# ============================================================================
# SECTION 1: Setup and Configuration
# ============================================================================

# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    DRIVE_MOUNTED = True
except ImportError:
    print("ℹ️ Not running in Colab - using local file system")
    DRIVE_MOUNTED = False

# EuroSAT class names
EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# ============================================================================
# SECTION 2: Model Architecture
# ============================================================================

class EuroSATResNet50(nn.Module):
    """
    ResNet50 model customized for EuroSAT classification

    Architecture:
    - Backbone: ResNet50 pretrained on ImageNet
    - Custom classifier head with dropout and batch normalization
    - Total parameters: ~25M
    """
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super(EuroSATResNet50, self).__init__()

        # Load pretrained ResNet50
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        num_features = self.backbone.fc.in_features  # 2048

        # Replace final FC layer with custom classifier
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

# ============================================================================
# SECTION 3: Image Preprocessing
# ============================================================================

def get_transforms():
    """
    Get image preprocessing transforms

    Transforms:
    1. Resize to 224x224 (ResNet50 input size)
    2. Convert to tensor
    3. Normalize with ImageNet mean and std
    """
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ])

# ============================================================================
# SECTION 4: Model Loading
# ============================================================================

def load_model(model_path):
    """
    Load ResNet50 model from checkpoint

    Args:
        model_path: Path to .pth model file

    Returns:
        model: Loaded model in eval mode
        info: Dictionary with model information
    """
    try:
        # Handle Google Drive paths
        if DRIVE_MOUNTED and not model_path.startswith('/content/drive/'):
            if not model_path.startswith('/'):
                model_path = '/content/drive/MyDrive/' + model_path

        # Load checkpoint
        print(f"📂 Loading model from: {model_path}")
        checkpoint = torch.load(model_path, map_location=device)

        # Create model
        model = EuroSATResNet50(num_classes=len(EUROSAT_CLASSES))

        # Load state dict
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            accuracy = checkpoint.get('test_acc', checkpoint.get('val_accuracy', 'N/A'))
            epoch = checkpoint.get('epoch', 'N/A')
        else:
            model.load_state_dict(checkpoint)
            accuracy = 'N/A'
            epoch = 'N/A'

        # Move to device and set to eval mode
        model.to(device)
        model.eval()

        # Model info
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        info = {
            'accuracy': accuracy,
            'epoch': epoch,
            'total_params': total_params,
            'trainable_params': trainable_params
        }

        print(f"✅ Model loaded successfully!")
        print(f"   Accuracy: {accuracy}%")
        print(f"   Epoch: {epoch}")
        print(f"   Parameters: {total_params:,}")

        return model, info

    except Exception as e:
        print(f"❌ Error loading model: {str(e)}")
        return None, {'error': str(e)}

# ============================================================================
# SECTION 5: Prediction Function
# ============================================================================

def predict(model, image, transform):
    """
    Make prediction on an image

    Args:
        model: Loaded PyTorch model
        image: PIL Image
        transform: Image preprocessing transforms

    Returns:
        results: Dictionary with prediction results
    """
    try:
        # Preprocess image
        image_tensor = transform(image).unsqueeze(0).to(device)

        # Make prediction
        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)

            # Get results
            probs = probabilities[0].cpu().numpy()
            predicted_idx = torch.argmax(outputs, dim=1).item()
            confidence = probabilities[0][predicted_idx].item()
            predicted_class = EUROSAT_CLASSES[predicted_idx]

            # Get top 5 predictions
            top5_indices = np.argsort(probs)[-5:][::-1]
            top5_classes = [EUROSAT_CLASSES[i] for i in top5_indices]
            top5_probs = [probs[i] for i in top5_indices]

            return {
                'predicted_class': predicted_class,
                'confidence': confidence,
                'all_probabilities': probs,
                'top5_classes': top5_classes,
                'top5_probs': top5_probs
            }

    except Exception as e:
        print(f"❌ Prediction error: {str(e)}")
        return None

# ============================================================================
# SECTION 6: Visualization
# ============================================================================

def create_bar_chart(results):
    """Create bar chart of top 5 predictions"""
    if results is None:
        return None

    fig, ax = plt.subplots(figsize=(10, 6))

    classes = results['top5_classes']
    probs = [p * 100 for p in results['top5_probs']]

    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(classes))]
    bars = ax.barh(classes, probs, color=colors)

    # Add percentage labels
    for i, (bar, prob) in enumerate(zip(bars, probs)):
        ax.text(prob + 1, bar.get_y() + bar.get_height()/2,
                f'{prob:.1f}%', va='center', fontweight='bold')

    ax.set_xlabel('Confidence (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Land Use Class', fontsize=12, fontweight='bold')
    ax.set_title('Top 5 Predictions', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 105)
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    return fig

# ============================================================================
# SECTION 7: Gradio Interface
# ============================================================================

# Global model variable
loaded_model = None
transform = get_transforms()

def load_model_gradio(model_path):
    """Load model for Gradio interface"""
    global loaded_model

    if not model_path or not model_path.strip():
        return "⚠️ Please provide a model path"

    loaded_model, info = load_model(model_path.strip())

    if loaded_model is None:
        return f"❌ Failed to load model: {info.get('error', 'Unknown error')}"

    return f"""✅ Model loaded successfully!

📊 Model Information:
• Accuracy: {info['accuracy']}%
• Epoch: {info['epoch']}
• Total Parameters: {info['total_params']:,}
• Trainable Parameters: {info['trainable_params']:,}

Ready to make predictions!"""

def predict_gradio(image):
    """Make prediction for Gradio interface"""
    global loaded_model

    if loaded_model is None:
        return None, "❌ Please load a model first", None

    if image is None:
        return None, "⚠️ Please provide an image", None

    # Make prediction
    results = predict(loaded_model, image, transform)

    if results is None:
        return None, "❌ Prediction failed", None

    # Create results text
    results_text = f"""✅ Prediction Complete!

🎯 Predicted Class: {results['predicted_class']}
📈 Confidence: {results['confidence']*100:.2f}%

📊 Top 5 Predictions:
"""
    for cls, prob in zip(results['top5_classes'], results['top5_probs']):
        results_text += f"• {cls}: {prob*100:.1f}%\n"

    # Create chart
    chart = create_bar_chart(results)

    return image, results_text, chart

def load_from_url(url):
    """Load image from URL"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image
    except Exception as e:
        return None

# Create Gradio interface
def create_interface():
    with gr.Blocks(theme=gr.themes.Soft(), title="EuroSAT ResNet50 Classifier") as demo:

        gr.Markdown("""
        # 🛰️ EuroSAT Land Use Classification with ResNet50

        Upload a satellite or aerial image to classify it into one of 10 land use categories.

        **Classes**: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial,
        Pasture, PermanentCrop, Residential, River, SeaLake
        """)

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🔧 Step 1: Load Model")

                model_path_input = gr.Textbox(
                    label="Model Path",
                    placeholder="/content/drive/MyDrive/weights/resnet50_eurosat_best.pth",
                    value="/content/drive/MyDrive/weights/resnet50_eurosat_best.pth",
                    info="Path to your ResNet50 model weights (.pth file)"
                )

                load_btn = gr.Button("📥 Load Model", variant="primary")
                model_status = gr.Textbox(
                    label="Model Status",
                    lines=8,
                    interactive=False
                )

                gr.Markdown("""
                ### 📝 Path Examples:
                - Google Drive: `/content/drive/MyDrive/weights/model.pth`
                - Relative: `weights/model.pth`
                - Local: `/path/to/model.pth`
                """)

            with gr.Column(scale=1):
                gr.Markdown("### 🖼️ Step 2: Upload Image")

                image_input = gr.Image(
                    label="Upload Image",
                    type="pil",
                    height=300
                )

                gr.Markdown("**Or load from URL:**")
                image_url = gr.Textbox(
                    label="Image URL",
                    placeholder="https://example.com/satellite-image.jpg"
                )
                load_url_btn = gr.Button("🔗 Load from URL")

                predict_btn = gr.Button("🔍 Classify Image", variant="primary", size="lg")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Results")
                results_text = gr.Textbox(
                    label="Prediction Results",
                    lines=12,
                    interactive=False
                )

            with gr.Column(scale=1):
                gr.Markdown("### 📈 Confidence Chart")
                results_chart = gr.Plot(label="Top 5 Predictions")

        gr.Markdown("""
        ---
        ### 📖 Instructions:
        1. **Load Model**: Enter the path to your trained ResNet50 model and click "Load Model"
        2. **Upload Image**: Either upload an image file or paste a URL
        3. **Classify**: Click "Classify Image" to see the prediction

        ### 🔬 Model Details:
        - **Architecture**: ResNet50 with custom classifier head
        - **Input Size**: 224×224 pixels (auto-resized)
        - **Output**: 10 land use classes
        - **Preprocessing**: ImageNet normalization
        """)

        # Event handlers
        load_btn.click(
            fn=load_model_gradio,
            inputs=[model_path_input],
            outputs=[model_status]
        )

        load_url_btn.click(
            fn=load_from_url,
            inputs=[image_url],
            outputs=[image_input]
        )

        predict_btn.click(
            fn=predict_gradio,
            inputs=[image_input],
            outputs=[image_input, results_text, results_chart]
        )

    return demo

# ============================================================================
# SECTION 8: Launch
# ============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("🛰️ EuroSAT LAND USE CLASSIFICATION - ResNet50")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Google Drive: {'✅ Mounted' if DRIVE_MOUNTED else '❌ Not mounted'}")
    print(f"Classes: {len(EUROSAT_CLASSES)}")
    print("=" * 70)

    # Create and launch interface
    demo = create_interface()
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        debug=True
    )

Mounted at /content/drive
✅ Google Drive mounted successfully!
🔧 Using device: cuda
🛰️ EuroSAT LAND USE CLASSIFICATION - ResNet50
Device: cuda
Google Drive: ✅ Mounted
Classes: 10
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6a5a1f2b7c434d15cc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📂 Loading model from: /content/drive/MyDrive/weights/resnet50_eurosat_best.pth
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 181MB/s]


✅ Model loaded successfully!
   Accuracy: 94.96972519795062%
   Epoch: 30
   Parameters: 24,692,042
